# At-Receipt Sell-Through Risk Model
### A Tool for Earlier, More Confident Inventory Decisions

---

**Prepared for:** Retail Merchants & Corporate Leadership  
**Date:** May 2026  
**Status:** Proof of Concept — Synthetic Data

In [ ]:
import pickle
import sqlite3
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
from pathlib import Path
from sklearn.model_selection import LeaveOneOut, cross_val_predict

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

DB        = Path("../sellthrough.db")
MODEL_PKL = Path("../models/model.pkl")

HEALTHY_THRESHOLD = 0.45
CAT_FEATURES = ["vendor", "department", "price_tier"]
NUM_FEATURES  = ["avg_retail_price", "receive_month", "units_received"]
FEATURES      = CAT_FEATURES + NUM_FEATURES

REC_ORDER  = ["Reorder", "Monitor", "Discount", "Return / Clear"]
REC_COLORS = {
    "Reorder":        "#2ca02c",
    "Monitor":        "#1f77b4",
    "Discount":       "#ff7f0e",
    "Return / Clear": "#d62728",
}

conn = sqlite3.connect(DB)
df = pd.read_sql_query("SELECT * FROM sku_sell_through", conn)
conn.close()

df["receive_month"] = pd.to_datetime(df["receive_date"]).dt.month
df["label"] = (df["sell_through_rate"] >= HEALTHY_THRESHOLD).astype(int)

with open(MODEL_PKL, "rb") as f:
    model = pickle.load(f)

X, y = df[FEATURES], df["label"]
loo = LeaveOneOut()
y_prob = cross_val_predict(model, X, y, cv=loo, method="predict_proba")[:, 1]

def recommend(p):
    if p >= 0.60: return "Reorder"
    if p >= 0.35: return "Monitor"
    if p >= 0.15: return "Discount"
    return "Return / Clear"

df["healthy_prob"]   = y_prob.round(3)
df["recommendation"] = df["healthy_prob"].apply(recommend)

---
## Executive Summary

Today, sell-through problems are discovered after they happen. Through markdowns, returns, and end-of-season write-offs. By then, the margin is already gone.

This model identifies which styles are likely to underperform **before a single unit sells** at the moment inventory is received. That window is where the most meaningful interventions are still available: early markdowns, transfer to higher-velocity doors, or coordinated promotional support.

Tested against 500 styles, **the model correctly identified 32 of 34 strong-performing styles at receipt** with no sales data, no weeks-of-supply calculation, and no manual review. The two it missed were identified as borderline cases.

The four outputs are direct and actionable: **Reorder, Monitor, Discount, or Return / Clear.**

---
## The Problem: Sell-Through Is a Selection Problem

The instinct when sell-through is low is to look at volume. Did we buy enough, price it right, put it in the right doors? Those are execution questions. But the data tells a different story.

**Most underperforming styles were never going to sell through at full price, regardless of how they were managed.** The problem was selected into the assortment before the first box arrived on the floor.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

sorted_df = df.sort_values("sell_through_rate", ascending=False).reset_index(drop=True)
colors = [REC_COLORS[r] for r in sorted_df["recommendation"]]

ax.bar(range(len(sorted_df)), sorted_df["sell_through_rate"], color=colors,
       edgecolor="none", width=1.0)
ax.axhline(HEALTHY_THRESHOLD, color="black", linestyle="--", linewidth=1,
           label=f"Healthy threshold ({HEALTHY_THRESHOLD:.0%})")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Styles (ranked by sell-through rate)")
ax.set_ylabel("Sell-Through Rate")
ax.set_title("Sell-Through Rate Across All 500 Styles")

patches = [mpatches.Patch(color=REC_COLORS[r], label=r) for r in REC_ORDER]
patches.append(mpatches.Patch(color="none", label=""))
ax.legend(handles=patches, loc="upper right", frameon=True)

plt.tight_layout()
plt.show()

pct_below = (df["sell_through_rate"] < HEALTHY_THRESHOLD).mean()
print(f"{pct_below:.0%} of styles sold through below the healthy threshold.")
print(f"The majority of the assortment is in Discount or Return / Clear territory.")

---
## What the Model Uses — and What It Doesn't

The model makes its prediction **at the moment inventory is received**, using only information already in hand:

| Signal | What it represents |
|---|---|
| Units received | Buyer confidence at time of order |
| Receive month | Seasonal timing |
| Retail price | Price tier positioning |
| Vendor | Brand |
| Department | Category |
| Price tier | Budget / Mid / Premium / Luxury |

**What it does not use:** sell-through history, weeks of supply, sales velocity, or any data that only becomes visible after selling begins. This is intentional, early action requires early signals.

---
## Key Finding 1: Buyer Confidence Is the Strongest Signal

The single most predictive input is how many units were ordered. Styles that buyers committed to in larger quantities at the time of order consistently outperformed those bought conservatively.

This is not a statement that buying more units *causes* better sell-through. It reflects something more useful: **buyers often know which styles have energy before they hit the floor.** That conviction, visible in the order quantity , is the model's strongest early signal.

The implication: when buyers hedge on a style, the model treats that hesitation as a risk flag.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for rec in REC_ORDER:
    sub = df[df["recommendation"] == rec]
    ax.scatter(sub["units_received"], sub["sell_through_rate"],
               label=rec, color=REC_COLORS[rec],
               alpha=0.6, edgecolors="none", s=30)

ax.axhline(HEALTHY_THRESHOLD, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Units Received at Receipt (buyer order quantity)")
ax.set_ylabel("Actual Sell-Through Rate")
ax.set_title("Buyer Order Quantity vs. Actual Sell-Through")
ax.legend(title="Recommendation", frameon=True)
plt.tight_layout()
plt.show()

---
## Key Finding 2: Premium Running Is Outperforming

Among the clearest patterns in the data: **ON, ASICS, and HOKA styles are consistently scoring in the higher-confidence range.** Across the assortment, these brands appear most often in the Reorder and Monitor buckets, not because of price or category alone, but because their combination of signals (brand, price tier, buy depth) aligns with what the model has learned from healthy performers.

This is a category trend signal worth noting at the assortment planning level: premium running is showing structural sell-through strength relative to other categories in this dataset.

In [ ]:
vendor_summary = (
    df.groupby("vendor")
      .agg(
          styles=("sku", "count"),
          avg_healthy_prob=("healthy_prob", "mean"),
          avg_sell_through=("sell_through_rate", "mean"),
          reorder_pct=("recommendation", lambda x: (x == "Reorder").mean()),
      )
      .sort_values("avg_healthy_prob", ascending=False)
      .reset_index()
)

PREMIUM_RUNNING = {"ON", "HOKA", "ASICS", "SAUCONY", "NEW BALANCE"}
bar_colors = [
    "#2ca02c" if v in PREMIUM_RUNNING else "#aec7e8"
    for v in vendor_summary["vendor"]
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].barh(vendor_summary["vendor"], vendor_summary["avg_healthy_prob"],
             color=bar_colors, edgecolor="none")
axes[0].set_xlabel("Average Healthy Probability Score")
axes[0].set_title("Brand Ranking by Healthy Probability")
axes[0].axvline(0.5, color="black", linestyle="--", linewidth=1, alpha=0.5)

green_patch = mpatches.Patch(color="#2ca02c", label="Premium Running")
blue_patch  = mpatches.Patch(color="#aec7e8", label="Other")
axes[0].legend(handles=[green_patch, blue_patch])

dept_summary = (
    df.groupby("department")
      .agg(
          styles=("sku", "count"),
          avg_healthy_prob=("healthy_prob", "mean"),
      )
      .sort_values("avg_healthy_prob", ascending=False)
      .reset_index()
)

axes[1].barh(dept_summary["department"], dept_summary["avg_healthy_prob"],
             color="#1f77b4", edgecolor="none")
axes[1].set_xlabel("Average Healthy Probability Score")
axes[1].set_title("Category Ranking by Healthy Probability")
axes[1].axvline(0.5, color="black", linestyle="--", linewidth=1, alpha=0.5)

plt.tight_layout()
plt.show()

---
## The Action Framework: Four Buckets

Every style receives a score from 0 to 1 representing its likelihood of healthy sell-through. That score maps directly to one of four recommended actions:

| Bucket | Score | What it means | Suggested action |
|---|---|---|---|
| **Reorder** | ≥ 0.60 | Strong sell-through expected | Secure replenishment; confirm depth with vendor |
| **Monitor** | 0.35 – 0.60 | On track, some uncertainty | Watch weekly; no action yet |
| **Discount** | 0.15 – 0.35 | At risk of not selling through | Plan promotional cadence early |
| **Return / Clear** | < 0.15 | High risk of excess inventory | Initiate return authorization or clearance plan |

> **Reorder caveat:** Limited release and exclusive product cannot always be reordered regardless of model output. For these styles, a Reorder score should be interpreted as *confirmation of demand strength* that's useful for negotiating future allocations or prioritizing floor placement, rather than a literal replenishment trigger.

In [ ]:
rec_counts = df["recommendation"].value_counts().reindex(REC_ORDER).fillna(0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bars = axes[0].bar(
    rec_counts.index, rec_counts.values,
    color=[REC_COLORS[r] for r in REC_ORDER],
    edgecolor="k", width=0.6,
)
for bar, count in zip(bars, rec_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
                 str(count), ha="center", va="bottom", fontweight="bold")
axes[0].set_title("Styles by Recommended Action (500 total)")
axes[0].set_ylabel("# Styles")
axes[0].set_xlabel("")
plt.sca(axes[0])
plt.xticks(rotation=15, ha="right")

# Sell-through distribution per bucket
df.boxplot(
    column="sell_through_rate", by="recommendation",
    positions=range(len(REC_ORDER)), ax=axes[1],
)
axes[1].set_xticklabels(REC_ORDER, rotation=15, ha="right")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].axhline(HEALTHY_THRESHOLD, color="black", linestyle="--", linewidth=1, alpha=0.5)
axes[1].set_title("Actual Sell-Through by Recommendation Bucket")
axes[1].set_xlabel("")
plt.sca(axes[1])
plt.title("Actual Sell-Through by Recommendation Bucket")

plt.tight_layout()
plt.show()

print(df.groupby("recommendation")["sell_through_rate"]
        .agg(["count", "mean", "median"])
        .reindex(REC_ORDER)
        .rename(columns={"count": "styles", "mean": "avg ST%", "median": "median ST%"})
        .round(3)
        .to_string())

---
## How Accurate Is This?

This is not a question that belongs in a model accuracy table. Here is what the testing actually showed, in plain terms:

**The model correctly identified 32 out of 34 strong-performing styles before a single unit sold.**

The two it missed were borderline: both sat just above the healthy threshold with actual sell-through rates of 45–47%, where real-world uncertainty would also make a human call difficult.

Put differently: if a merchant had used these scores to make early reorder decisions on the day inventory was received, they would have been right 94% of the time on the styles that actually warranted reorder action.

The model also generates false positives and styles it scored as healthy that ultimately underperformed. In a live deployment, these would be surfaced as "Monitor" rather than "Reorder," where weekly tracking can catch the correction early.

In [ ]:
y_pred = (y_prob >= 0.5).astype(int)

tp = int(((y == 1) & (y_pred == 1)).sum())
fn = int(((y == 1) & (y_pred == 0)).sum())
fp = int(((y == 0) & (y_pred == 1)).sum())
tn = int(((y == 0) & (y_pred == 0)).sum())

fig, ax = plt.subplots(figsize=(7, 3))
ax.axis("off")

table_data = [
    ["", "Model said: Reorder", "Model said: At-Risk"],
    ["Actually healthy",  f"{tp} correctly flagged for reorder",  f"{fn} missed"],
    ["Actually at-risk",  f"{fp} false reorder flags",             f"{tn} correctly identified"],
]

tbl = ax.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    loc="center",
    cellLoc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.4, 2.2)

tbl[(1, 1)].set_facecolor("#d4edda")
tbl[(1, 2)].set_facecolor("#f8d7da")
tbl[(2, 1)].set_facecolor("#fff3cd")
tbl[(2, 2)].set_facecolor("#d4edda")

ax.set_title("Model Results in Plain Terms  (tested against 500 styles)",
             fontsize=12, pad=20, loc="left")
plt.tight_layout()
plt.show()

---
## Limitations — Read Before Acting on This Model

This section is not a footnote. The limitations below are conditions that must be addressed before this model is used in any live buying or inventory decision.

---

### 1. This model was trained on synthetic data

Every style, sale, and sell-through rate in this dataset was computer-generated to simulate realistic retail patterns. The model has never seen real purchase orders, real consumer demand, or real store-level variation. Its accuracy figures, including the 32-of-34 result cited above, reflect performance on data that was constructed by the same logic the model learned from.

**What this means in practice:** Treat this as a proof of concept, not a production tool. The architecture is sound; the accuracy numbers are illustrative, not guaranteed.

---

### 2. Real deployment requires integration with receiving systems

For the model to fire at receipt (its core value proposition) it needs access to purchase order data at the store or DC level at the moment inventory is processed. That means integration with the WMS, PO management system, or ERP, depending on where receiving data lives.

A bolt-on reporting layer that runs weekly after the fact is not the same as an at-receipt signal. Latency in the trigger reduces the intervention window.

---

### 3. The healthy/at-risk threshold needs calibration against actual markdown windows

The 45% sell-through threshold used in this model was set based on the synthetic data distribution. In a real deployment, this number should be set in collaboration with merchants: **at what sell-through rate does a style actually need intervention, given your markdown calendar, return windows, and margin targets?**

A threshold set too high will generate excessive reorder flags. Too low, and the model misses styles that are quietly heading toward clearance. This calibration is a business decision, not a technical one.

---

### Summary of requirements for live deployment

| Requirement | Status |
|---|---|
| Train on real historical PO + sales data | Not started |
| Integration with receiving / WMS at store or DC | Not started |
| Threshold calibration with merchant team | Not started |
| Handling for limited release / exclusive SKUs | Not started |
| Ongoing model monitoring and retraining cadence | Not started |